In [2]:
import torch 
import numpy
import torch.nn as nn
import torch.nn.functional as F

d:\Application\Anaconda\envs\MSA102\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


embedding

In [3]:
batch_size = 2

# 单词表大小
max_num_src_words = 8
max_num_tgt_words = 8
model_dim = 8 

# src_len =torch.randint(2,5,(batch_size,))
# tgt_len = torch.randint(2,5,(batch_size,))
src_len = torch.Tensor([2,4]).to(torch.int32)
tgt_len = torch.Tensor([4,3]).to(torch.int32)

# 序列最大长度
max_src_seq_len = 5
max_tgt_seq_len = 5
max_pos_len = 5

# 单词索引构成的句子 并将其pad,然后变为2维,并连接
src_seq = torch.cat([torch.unsqueeze(F.pad(torch.randint(1, max_num_src_words, (L,)), (0, max_src_seq_len - L)) ,0) for L in src_len] )
tgt_seq = torch.cat([torch.unsqueeze(F.pad(torch.randint(1, max_num_tgt_words, (L,)), (0, max_tgt_seq_len - L)) ,0) for L in tgt_len] )



print(src_len)
print(tgt_len)

print(src_seq)
print(tgt_seq)

tensor([2, 4], dtype=torch.int32)
tensor([4, 3], dtype=torch.int32)
tensor([[2, 2, 0, 0, 0],
        [1, 4, 3, 4, 0]])
tensor([[7, 2, 7, 5, 0],
        [1, 3, 4, 0, 0]])


word embedding

In [4]:

# 构造word embedding
src_embedding_table = nn.Embedding(max_num_src_words +1, model_dim)
tgt_embedding_table = nn.Embedding(max_num_tgt_words +1, model_dim)
# 相当于给每个单词(单词表个数)都转换为向量表示，每个单词都是model_dim维
src_embedding = src_embedding_table(src_seq)
tgt_embedding = tgt_embedding_table(tgt_seq)
# 按照索引来确定该单词的向量表示
print(src_embedding_table)
print(src_embedding_table.weight)
print(src_seq)
# print(tgt_embedding_table.weight)
print(src_embedding)

Embedding(9, 8)
Parameter containing:
tensor([[-0.3221,  0.7901,  1.0483, -0.0246,  0.7544, -0.2590, -0.9289, -1.1973],
        [-0.6829, -0.9833,  1.7172,  1.3488,  0.0939, -0.3671,  0.1511, -0.4239],
        [-0.4096, -0.2593,  0.1545, -0.8072, -0.0791,  0.9279,  1.3047,  0.4103],
        [-0.4536, -0.1503,  0.4709, -0.7982,  0.8419,  1.4621, -0.1064, -1.2649],
        [-0.6299, -1.0277,  1.4965, -0.7519, -0.5309,  0.2998,  0.5315,  0.3084],
        [-0.5667,  0.4411,  0.3157, -1.2092, -0.9289,  1.9229, -1.5017,  0.0619],
        [-1.6518,  1.7497, -0.0722,  1.4441, -0.1629,  0.5576, -1.0194, -0.1565],
        [ 0.8974, -1.6389, -0.3276,  0.4948,  0.5039,  1.6688,  0.2155, -0.9904],
        [-0.2597,  0.7295,  0.1577, -1.2887,  0.6733,  0.0869,  1.6403,  1.3465]],
       requires_grad=True)
tensor([[2, 2, 0, 0, 0],
        [1, 4, 3, 4, 0]])
tensor([[[-0.4096, -0.2593,  0.1545, -0.8072, -0.0791,  0.9279,  1.3047,
           0.4103],
         [-0.4096, -0.2593,  0.1545, -0.8072, -0.079

position embedding

In [5]:
# 构造position embedding

pos_mat = torch.arange(max_pos_len).reshape((-1,1))
i_mat = torch.pow(10000, torch.arange(0,8,2).reshape((1,-1))/model_dim)
# 0 2 4 8
pe_embedding_table = torch .zeros(max_pos_len , model_dim)
pe_embedding_table[:,0::2] = torch.sin(pos_mat / i_mat)
pe_embedding_table[:,1::2] = torch.cos(pos_mat / i_mat)
pe_embedding_table
# 一共有五行，每一行都表示一个位置0-4
# 一共有八列，表示单词向量维度？？

pe_embedding = nn.Embedding(max_pos_len, model_dim)
pe_embedding.weight = nn.Parameter(pe_embedding_table, requires_grad=False)

src_pos = torch.cat([torch.unsqueeze(torch.arange(max(src_len)),0) for _ in src_len]).to(torch.int32)
tgt_pos = torch.cat([torch.unsqueeze(torch.arange(max(tgt_len)),0) for _ in src_len]).to(torch.int32)
src_pe_embedding = pe_embedding(src_pos)
tgt_pe_embedding = pe_embedding(tgt_pos)
print(src_pe_embedding)

tensor([[[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
           1.0000e+00,  0.0000e+00,  1.0000e+00],
         [ 8.4147e-01,  5.4030e-01,  9.9833e-02,  9.9500e-01,  9.9998e-03,
           9.9995e-01,  1.0000e-03,  1.0000e+00],
         [ 9.0930e-01, -4.1615e-01,  1.9867e-01,  9.8007e-01,  1.9999e-02,
           9.9980e-01,  2.0000e-03,  1.0000e+00],
         [ 1.4112e-01, -9.8999e-01,  2.9552e-01,  9.5534e-01,  2.9995e-02,
           9.9955e-01,  3.0000e-03,  1.0000e+00]],

        [[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
           1.0000e+00,  0.0000e+00,  1.0000e+00],
         [ 8.4147e-01,  5.4030e-01,  9.9833e-02,  9.9500e-01,  9.9998e-03,
           9.9995e-01,  1.0000e-03,  1.0000e+00],
         [ 9.0930e-01, -4.1615e-01,  1.9867e-01,  9.8007e-01,  1.9999e-02,
           9.9980e-01,  2.0000e-03,  1.0000e+00],
         [ 1.4112e-01, -9.8999e-01,  2.9552e-01,  9.5534e-01,  2.9995e-02,
           9.9955e-01,  3.0000e-03,  1.0000e+00]

encoder attetion Mask

In [6]:
valid_encoder_pos = torch.unsqueeze(torch.cat([torch.unsqueeze(F.pad(torch.ones(L), (0,max(src_len) - L)), 0) for L in src_len]), 2)
# 2,4,1
valid_encoder_pos_matrix = torch.bmm(valid_encoder_pos, valid_encoder_pos.transpose(1,2))
invalid_encoder_pos_matrix = 1- valid_encoder_pos_matrix
mask_encoder_Self_attention = invalid_encoder_pos_matrix.to(torch.bool)

mask_encoder_Self_attention
# 2,4,4
# 用来标志某个单词和某个单词之间是否有关联

tensor([[[False, False,  True,  True],
         [False, False,  True,  True],
         [ True,  True,  True,  True],
         [ True,  True,  True,  True]],

        [[False, False, False, False],
         [False, False, False, False],
         [False, False, False, False],
         [False, False, False, False]]])

intra-attention Mask

In [7]:
# Q K^T [batchsize, tgt_seq_len, src_seq_len]

valid_decoder_pos = torch.unsqueeze(torch.cat([torch.unsqueeze(F.pad(torch.ones(L), (0,max(tgt_len) - L)), 0) for L in tgt_len]), 2)
valid_cross_pos_matrix = torch.bmm(valid_decoder_pos, valid_encoder_pos.transpose(1,2))
# 表示src和tgt之间单词的有效性
invalid_cross_pos_matrix = 1 - valid_cross_pos_matrix
mask_cross_attention = invalid_cross_pos_matrix.to(torch.bool)



decoder self-attetion mask

In [8]:
valid_decoder_tri_matrix = torch.cat([torch.unsqueeze(F.pad(torch.tril(torch.ones((L,L))),(0,max(tgt_len)-L,0,max(tgt_len)-L) ),0)for L in tgt_len],0)
invalid_decoder_tri_matrix = 1 -valid_decoder_tri_matrix
invalid_decoder_tri_matrix = invalid_decoder_tri_matrix.to(torch.bool)
invalid_decoder_tri_matrix

tensor([[[False,  True,  True,  True],
         [False, False,  True,  True],
         [False, False, False,  True],
         [False, False, False, False]],

        [[False,  True,  True,  True],
         [False, False,  True,  True],
         [False, False, False,  True],
         [ True,  True,  True,  True]]])

In [9]:
score = torch.randn(batch_size, max(tgt_len), max(tgt_len))
masked_score = score.masked_fill(invalid_decoder_tri_matrix, -1e9)
prob = F.softmax(masked_score,-1)
prob


tensor([[[1.0000, 0.0000, 0.0000, 0.0000],
         [0.2054, 0.7946, 0.0000, 0.0000],
         [0.5230, 0.4428, 0.0342, 0.0000],
         [0.2106, 0.2912, 0.0944, 0.4037]],

        [[1.0000, 0.0000, 0.0000, 0.0000],
         [0.4050, 0.5950, 0.0000, 0.0000],
         [0.2180, 0.5976, 0.1845, 0.0000],
         [0.2500, 0.2500, 0.2500, 0.2500]]])

Self-attetion

In [10]:
def scaled_dot_product_attention(Q, K, V, attn_mask):
    score = torch.bmm(Q,K.transpose(-2,-1))/torch.sqrt(model_dim)
    masked_score = score.masked_fill(attn_mask, -1e9)
    prob = F.softmax(masked_score, -1)
    context = torch.bmm(prob, V)
    return context